# Prepare contract/category data

Reshape CUAD's question-answer JSON into one record per contract and clause category. Each record includes whether the category is present and all annotated answer spans. The official train and test splits stay separate.


In [ ]:
import json
from pathlib import Path
import pandas as pd


## Locate and load the source data

This works whether the notebook is launched from the repository root or from the `notebooks` directory.


In [ ]:
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(path for path in candidates if (path / 'data/cuad/train_separate_questions.json').exists())
data_dir = project_root / 'data' / 'cuad'

with (data_dir / 'train_separate_questions.json').open() as f:
    train = json.load(f)
with (data_dir / 'test.json').open() as f:
    test = json.load(f)

print('Project root:', project_root)
print('Train contracts:', len(train['data']))
print('Test contracts:', len(test['data']))


## Build one record per contract/category

The category is quoted in each question. Multiple QA entries may name the same category, so their answers are combined. Answer text and offsets are retained. Categories with no answers still receive a record with `present=False` and an empty answer list.


In [ ]:
def get_category(question):
    parts = question.split('"')
    if len(parts) < 2:
        raise ValueError(f'Could not extract category from question: {question!r}')
    return parts[1]

def transform_split(dataset, split_name):
    records = []
    for doc in dataset['data']:
        paragraph = doc['paragraphs'][0]
        grouped_answers = {}
        for qa in paragraph['qas']:
            category = get_category(qa['question'])
            answers = grouped_answers.setdefault(category, [])
            for answer in qa['answers']:
                start = answer.get('answer_start')
                answers.append({
                    'text': answer['text'],
                    'answer_start': start,
                    'answer_end': start + len(answer['text']) if start is not None else None,
                })

        for category, answers in grouped_answers.items():
            records.append({
                'split': split_name,
                'contract_id': doc['title'],
                'contract_title': doc['title'],
                'category': category,
                'present': bool(answers),
                'answers': answers,
            })
    return records

records = transform_split(train, 'train') + transform_split(test, 'test')
df = pd.DataFrame(records)
df.head()


## Validate the representation

There should be one row for every official contract/category pair. The checks below also ensure there are no duplicate pairs and that presence agrees with whether answers exist.


In [ ]:
expected_contracts = len(train['data']) + len(test['data'])
category_counts = df.groupby('split')['category'].nunique()
rows_per_contract = df.groupby(['split', 'contract_id']).size()

assert df[['split', 'contract_id', 'category']].duplicated().sum() == 0
assert len(rows_per_contract) == expected_contracts
assert rows_per_contract.nunique() == 1
assert (df['present'] == df['answers'].map(bool)).all()
assert (category_counts == 41).all(), category_counts.to_dict()

print('Contracts:', expected_contracts)
print('Categories per contract:', int(rows_per_contract.iloc[0]))
print('Rows:', len(df), '(expected', expected_contracts * 41, ')')
print('Rows by split:')
print(df.groupby('split').size())


## Save as JSON Lines

JSONL keeps `answers` as a proper list of objects, including multiple spans and their offsets. The output is written to `data/processed/contract_categories.jsonl`.


In [ ]:
output_dir = project_root / 'data' / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'contract_categories.jsonl'
df.to_json(output_path, orient='records', lines=True, force_ascii=False)
print(f'Saved {len(df):,} records to {output_path}')
